# COMP0267 Coursework: compositional text–image models, from tensor networks to quantum hardware

In this coursework the meaning of a caption is computed from the meanings of its words, **wired together by its
grammar**. The same wiring can be evaluated as a classical **tensor network** or as a **parametrised quantum
circuit**. You will train both, and then make the quantum model's circuits run well on emulated (and, optionally,
real) IBM and IQM quantum computers. That last part carries most of the marks.

The data is **SVO-Probes** (Hendricks & Nematzadeh, 2021): about 9,000 *subject–verb–object* captions, each paired
with a **positive** image that the caption describes and a **negative** image that differs from it in exactly one of
the three. For example (Section 0 shows the two images):

| caption | positive image shows | negative image shows | differs in |
|---|---|---|---|
| *A woman holds a cat.* | (woman, hold, cat) | (woman, hold, dog) | the object |

We evaluate two two-way choices:

* **image choice**: given the caption *A woman holds a cat.*, pick the positive image over the negative one.
* **caption choice** (**SVO-Swap**): given the positive image, pick *A woman holds a cat.* over the same caption with
  subject and object swapped, *A cat holds a woman.* Both captions contain exactly the same words, so only the way
  the words are composed can tell them apart.

Images are represented by their (frozen) CLIP ViT-B/32 embeddings, and only the text side learns.

### How to work and what to submit
* Open the notebook in Google Colab and run it from top to bottom. The first cell installs the course package, and
  every provided cell runs as is. See the repository's README for a step-by-step setup guide.
* The provided models and settings are deliberately **minimal starting points**: they are poor, and improving them is
  part of the coursework.
* Questions are numbered **Q1–Q13**, with marks in brackets (**100 marks** in total). Answer in the cells marked
  **Answer** (markdown for discussion, code cells where asked). Keep answers concise: a
  table or plot, and two or three sentences of interpretation is usually enough.
* The course code is the `comp0267` package, a minimal skeleton. Do not edit it: build on it in this notebook by
  importing its classes and subclassing them (Part 3 does this with `Ansatz`).
* Section 6 (real quantum hardware) is optional and not marked.

| Part | Topic | Marks |
|---|---|---|
| 1 | Sentences as diagrams and tensor networks | 10 |
| 2 | Training the tensor network | 15 |
| 3 | Training the quantum model | 20 |
| 4 | Running the circuits on emulated hardware | 45 |
| 5 | Discussion | 10 |

## 0. Setup
The cell below installs the course package `comp0267` with its requirements (about two minutes on Colab), and
`setup()` prepares the runtime. Run it again whenever Colab starts a new session.

In [ ]:
%pip install -q "git+https://github.com/quantum-learning-labs/COMP0267_CW@main"
from comp0267.utils import setup

setup()

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from comp0267.utils import get_device, set_seed

SEED, BATCH_SIZE = 0, 256
set_seed(SEED)
device = get_device()
print(f"device: {device}")

`load_svo_probes` downloads the processed data on first use (15 MB, cached): every row's caption, image ids and URLs,
parsed diagram and split, and the CLIP embedding of every image.

In [ ]:
from comp0267.data.svo_probes import load_svo_probes, show_example

df, embeddings = load_svo_probes()
train, val, test = (df[df["split"] == split].reset_index(drop=True) for split in ("train", "val", "test"))
print(f"train {len(train)} | val {len(val)} | test {len(test)} rows, {len(embeddings)} image embeddings")
df[["corrected_sentence", "pos_triplet", "neg_triplet", "subj_neg", "verb_neg", "obj_neg"]].head()

The entry from the introduction with its two images. The models never see the pixels, only the images' CLIP
embeddings. The images are fetched from their original web pages, and some of those have gone offline since 2021.

In [ ]:
show_example(train[(train["corrected_sentence"] == "A woman holds a cat.") & (train["neg_triplet"] == "woman,hold,dog")].iloc[0])

## Part 1. Sentences as diagrams and tensor networks (10 marks)

### 1.1 Grammar as wiring
A **combinatory categorial grammar (CCG)** parser assigns every word a type built from atomic types such as $n$ (noun)
and $s$ (sentence). A transitive verb has type $(s\backslash n)/n$: *give me a noun on the right and a noun on the
left and I return a sentence*. Translating each type into a list of **wires**, one per atomic type, gives a transitive
verb three wires $(n, s, n)$ and a noun one wire $(n)$. Grammatical reduction then **connects** the subject's wire to
the verb's left $n$ wire and the object's wire to its right $n$ wire. The verb's $s$ wire is left open: it carries
the meaning of the sentence.

The result is a **string diagram**, stored as a list of boxes `(word, wires, types)`. Two boxes that share a wire
label are connected, and wire `0` is the sentence's output. Here is a training caption, parsed by the Bobcat parser:

In [ ]:
from comp0267.symbolic.diagrams import add_root

example = train.iloc[0]
print(example["corrected_sentence"])
for word, wires, types in example["corrected_sentence_diagram"]:
    print(f"  {word:>8}  wires {str(wires):12} types {types}")

The classical model maps every sentence to the same output space with a shared **`ROOT`** box: `add_root` plugs the
open wires of the *head* box (here the verb) into `ROOT`, whose last wire, of type `out`, is the sentence's output.

In [ ]:
for word, wires, types in add_root(example["corrected_sentence_diagram"]):
    print(f"  {word:>8}  wires {str(wires):12} types {types}")

The data's diagrams were produced by `parse`, and you can parse your own sentences. `load_parser` downloads the Bobcat
model on first use (about 0.8 GB, cached for the session).

In [ ]:
from comp0267.symbolic.diagrams import load_parser, parse

parser = load_parser()
for word, wires, types in parse(["A dog chases a red ball."], parser)[0]:
    print(f"  {word:>8}  wires {str(wires):12} types {types}")

**Q1 (5 marks).** Take the caption above with its `ROOT` box, and let every wire of type $t$ be a vector-space index
of dimension $d_t$ (a box with types $(n, s, n)$ is a tensor in $\mathbb{R}^{d_n \times d_s \times d_n}$).
(a) Give the shape of every tensor. (b) Write the sentence vector as an explicit sum over indices and say which sums
are the connections of the diagram. (c) How does the model share parameters across sentences?

**Answer Q1.** *Write your answer here.*

### 1.2 Compiling diagrams into tensor networks
`TNCompiler` turns a diagram into a **recipe**: an `einsum` string plus one `Op` per tensor, named by the word's
*symbol* (`word_types`). With `bond_dim` set, boxes with more than two wires (verbs, `ROOT`) are stored as a **matrix product state** (MPS): a chain
of small cores joined by bond wires of dimension $\chi$.

In [ ]:
from comp0267.compilation.tensor_network import TNCompiler
from comp0267.models.text import TNModel

DIMS = {"n": 1, "s": 1, "p": 1, "out": 64}  # wire dimension per type, `out` is the number of image features
BOND_DIM = 1
tn_compiler = TNCompiler(DIMS, bond_dim=BOND_DIM)

recipe = tn_compiler(example["corrected_sentence_diagram"])
print(recipe.einsum)
for op in recipe.ops:
    print("  ", op)

for split in (train, val, test):
    split["tn"] = split["corrected_sentence_diagram"].map(tn_compiler)

**Q2 (5 marks).** Measure what the tensor network costs. (a) Count the parameters of a verb with types $(n, s, n)$
as a full tensor and as an MPS (`TNCompiler(..., bond_dim=...)`) for a few wire and bond dimensions, and plot both.
(b) Time a forward pass of `TNModel` over a batch of captions as the wire dimensions grow. Which part of the diagram
dominates the cost, and why does contracting these diagrams stay cheap?

**Answer Q2.** *Write your answer here.*

## Part 2. Training the tensor network (15 marks)

### 2.1 The image side and the loss
The image side is frozen: we keep the `DIMS["out"]` CLIP dimensions with the highest variance over the training
images.

We train with **InfoNCE**. For a batch of $B$ matched pairs $(t_i, v_i)$ with similarities $s_{ij} = \cos(t_i, v_j)$,
each row of the similarity matrix is a $B$-way classification (*which image belongs to caption $i$?*), and so is each
column:
$$
\mathcal{L} = -\frac{1}{2B}\sum_{i=1}^{B}\left[\log\frac{e^{s_{ii}/\tau}}{\sum_{j} e^{s_{ij}/\tau}}
            + \log\frac{e^{s_{ii}/\tau}}{\sum_{j} e^{s_{ji}/\tau}}\right].
$$

In [ ]:
from comp0267.data.svo_probes import ChoiceDataset, collate, svo_swap
from comp0267.models.training import InfoNCE, choice_scores, cosine, fidelity

train_images = torch.stack([embeddings[int(i)] for i in train["pos_image_id"].unique()])
image_variance = train_images.var(0).to(device)  # variance of each CLIP dimension over the training images


def tn_image_model(images):
    dims = image_variance.topk(DIMS["out"]).indices  # the highest-variance CLIP dimensions
    return images.to(device)[:, dims]


train_loader = DataLoader(ChoiceDataset(train["tn"], train["pos_image_id"], embeddings, train["neg_image_id"]),
                          batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

### 2.2 Training

In [ ]:
TN_EPOCHS, TN_LR = 1, 1e-3

set_seed(SEED)
tn_model = TNModel(train["tn"]).to(device)
loss_fn = InfoNCE(cosine)
optimizer = torch.optim.Adam(tn_model.parameters(), lr=TN_LR)
for epoch in range(TN_EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        loss = loss_fn(tn_model(batch["caption"]), tn_image_model(batch["image"]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch["caption"])
    print(f"TN epoch {epoch + 1}/{TN_EPOCHS} | loss {total_loss / len(train_loader.dataset):.4f}")

### 2.3 Evaluation
`choice_scores` scores each row 1 if the positive scores higher than the negative, ½ on a tie and 0 otherwise.
Captions with a symbol the model never saw in training cannot be evaluated, so we keep the rows whose captions use
only known symbols. `svo_swap` rebuilds **SVO-Swap**, the 95-pair benchmark of Lo et al. (2025), from the val and
test rows: captions whose subject and object are both people or animals, with the two swapped.

In [ ]:
swap_test = svo_swap(df)
swap_test["tn"] = swap_test["corrected_sentence_diagram"].map(tn_compiler)
swap_test["tn_swapped"] = swap_test["swapped_sentence_diagram"].map(tn_compiler)

rows = test[test["tn"].map(tn_model.knows)]
image_loader = DataLoader(ChoiceDataset(rows["tn"], rows["pos_image_id"], embeddings, rows["neg_image_id"], "image"),
                          batch_size=BATCH_SIZE, collate_fn=collate)
rows = swap_test[swap_test["tn"].map(tn_model.knows) & swap_test["tn_swapped"].map(tn_model.knows)]
swap_loader = DataLoader(ChoiceDataset(rows["tn"], rows["pos_image_id"], embeddings, rows["tn_swapped"], "caption"),
                         batch_size=BATCH_SIZE, collate_fn=collate)

print(f"TN image choice: {choice_scores(tn_model, tn_image_model, image_loader, cosine).mean():.4f}")
print(f"TN SVO-Swap:     {choice_scores(tn_model, tn_image_model, swap_loader, cosine).mean():.4f}")
print(f"TN parameters:   {sum(p.numel() for p in tn_model.parameters())}")

**Q3 (5 marks).** Explain the accuracies above by looking inside the model. (a) Compute the caption vectors of a few
test captions and of their SVO-Swap partners, and compare them (e.g. their pairwise cosine similarities). What do
you see, and why does it follow from $d_n = d_s = d_p = 1$? (b) What does that mean for each choice? Why is the
SVO-Swap score close to, but not exactly, 0.5?

**Answer Q3.** *Write your answer here.*

**Q4 (10 marks).** Improve the tensor network. Tune at least the wire dimensions, the bond dimension, the learning rate
and the number of epochs, choosing settings on the **validation** split (`val`) and reporting the **test** accuracy
of your final choice. Give a table of your main runs (settings, parameters, validation and test accuracy on both
choices) and explain what limits each choice. Keep the cost of training in mind.

*Things to know.* The caption vector is `ROOT` applied to the head word's $s$ wire, so every caption vector lies in
a subspace of dimension at most $d_s$ of the `DIMS["out"]`-dimensional output: the images can only be told apart
along those directions. An MPS with bond dimension $\chi$ caps the rank of every verb at $\chi$. One epoch at a
learning rate of $10^{-3}$ barely moves the initial tensors: compare the printed loss with its value at chance,
$\log B$ for a batch of $B$ pairs.

In [ ]:
# Q4: your code here

**Answer Q4.** *Write your answer here.*

## Part 3. Training the quantum model (20 marks)

### 3.1 From diagrams to circuits
The same diagrams can be evaluated on a quantum computer:
* Every wire of type $t$ becomes $q_t$ **qubits**.
* Every word becomes a **parametrised circuit** (an *ansatz*), applied from the leaves of the diagram to its head.
  A word with no arguments (a noun) acts on fresh qubits in $|0\dots0\rangle$. A word with arguments acts on the
  qubits its arguments already hold, plus fresh qubits for any wire it creates (a transitive verb adds its $s$
  qubits). A word that only modifies its argument (a determiner, an adjective) reuses the argument's qubits.
* The argument qubits left over at the end are measured and **post-selected** on 0.

The text state $|t\rangle = \big(\bigotimes_{\text{leftover}} \langle 0|\big)\, U_{\text{head}} \cdots
U_{\text{leaves}} |0\dots0\rangle$ is read off the head word's $s$ qubits (there is no `ROOT` box), and it is
*unnormalised*.
`comp0267.compilation.ansatz.Ansatz` does the wiring, and a subclass only says which gates a word applies to its qubits. The
`BareAnsatz` below applies one $R_y$ rotation per qubit, then a ladder of CNOTs, per layer.

In [ ]:
from comp0267.compilation.ansatz import Ansatz
from comp0267.execution.circuits import to_circuit
from comp0267.models.text import VQCModel
from comp0267.models.vision import FeatureMap


class BareAnsatz(Ansatz):
    def word_circuit(self, circuit, wires, name):
        for layer in range(self.layers):
            for q in range(len(wires)):
                circuit.apply("Ry", wires, [q], f"{name}_l{layer}_q{q}")
            for q in range(len(wires) - 1):
                circuit.apply("CX", wires, [q, q + 1])


QUBITS, LAYERS = {"n": 1, "s": 1, "p": 1}, 1
ansatz = BareAnsatz(QUBITS, layers=LAYERS)
for split in (train, val, test):
    split["vqc"] = split["corrected_sentence_diagram"].map(ansatz)

set_seed(SEED)
vqc_model = VQCModel(train["vqc"]).to(device)
example_circuit = to_circuit(train["vqc"].iloc[0], vqc_model.angle_dict())
print(train["corrected_sentence"].iloc[0], "|", example_circuit.metadata)
example_circuit.draw(fold=-1)

**Q5 (5 marks).** Look at the circuits. (a) Draw the circuits of a caption with a determiner or an adjective and of
its SVO-Swap partner, and say which qubits each word acts on and which are post-selected. (b) Plot how many qubits
and post-selected qubits the training captions need, and say which words add qubits and which do not.

**Answer Q5.** *Write your answer here.*

### 3.2 The image side
The image becomes a state on the same number of qubits as the sentence wire. `FeatureMap` works like `Ansatz`: a
subclass says how a CLIP embedding becomes angles (`map`) and which gates use them (`image_circuit`, where the
angle named `x{j}` is column `j` of `map`). `BareFeatureMap`
keeps the highest-variance CLIP dimensions and applies $R_y(x)$ to every qubit in every layer.

In [ ]:
class BareFeatureMap(FeatureMap):
    def map(self, images):
        dims = image_variance.topk(self.n_features).indices  # the highest-variance CLIP dimensions
        return images[:, dims.to(images.device)]

    def image_circuit(self, circuit, wires):
        for layer in range(self.layers):
            for q in range(len(wires)):
                circuit.apply("Ry", wires, [q], f"x{layer * len(wires) + q}")


vqc_image_model = BareFeatureMap(n_qubits=QUBITS["s"], layers=1).to(device)

**Q6 (4 marks).** (a) Show that $R_y(a) R_y(b) = R_y(a+b)$, so stacking layers of `BareFeatureMap` on one qubit adds
no expressive power. Which one-qubit states can it reach? (b) Plot the angles `map` gives the training images and the
image states they produce. What does that mean for telling images apart, and what would you change?

**Answer Q6.** *Write your answer here.*

### 3.3 Training by exact simulation
For states the natural similarity is the **fidelity** $|\langle \hat t | v \rangle|^2$ with $\hat t = t/\lVert t\rVert$.
`VQCModel` contracts the circuits' tensors exactly (a *perfect simulation*), so gradients are exact.

In [ ]:
VQC_EPOCHS, VQC_LR = 1, 1e-3

vqc_loader = DataLoader(ChoiceDataset(train["vqc"], train["pos_image_id"], embeddings, train["neg_image_id"]),
                        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
loss_fn = InfoNCE(fidelity)
optimizer = torch.optim.Adam(vqc_model.parameters(), lr=VQC_LR)
for epoch in range(VQC_EPOCHS):
    total_loss = 0.0
    for batch in vqc_loader:
        loss = loss_fn(vqc_model(batch["caption"]), vqc_image_model(batch["image"]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch["caption"])
    print(f"VQC epoch {epoch + 1}/{VQC_EPOCHS} | loss {total_loss / len(vqc_loader.dataset):.4f}")

In [ ]:
swap_test["vqc"] = swap_test["corrected_sentence_diagram"].map(ansatz)
swap_test["vqc_swapped"] = swap_test["swapped_sentence_diagram"].map(ansatz)

rows = test[test["vqc"].map(vqc_model.knows)]
image_loader = DataLoader(ChoiceDataset(rows["vqc"], rows["pos_image_id"], embeddings, rows["neg_image_id"], "image"),
                          batch_size=BATCH_SIZE, collate_fn=collate)
rows = swap_test[swap_test["vqc"].map(vqc_model.knows) & swap_test["vqc_swapped"].map(vqc_model.knows)]
swap_loader = DataLoader(ChoiceDataset(rows["vqc"], rows["pos_image_id"], embeddings, rows["vqc_swapped"], "caption"),
                         batch_size=BATCH_SIZE, collate_fn=collate)

print(f"VQC image choice: {choice_scores(vqc_model, vqc_image_model, image_loader, fidelity).mean():.4f}")
print(f"VQC SVO-Swap:     {choice_scores(vqc_model, vqc_image_model, swap_loader, fidelity).mean():.4f}")
print(f"VQC parameters:   {vqc_model.angles.numel()}")

**Q7 (11 marks).** Improve the quantum model by exact simulation. Change the ansatz (a new `Ansatz` subclass: other
rotations, entanglers, layers), the qubits per type in `QUBITS`, the image feature map (a `FeatureMap` subclass whose
qubits match `QUBITS["s"]`) and the training settings. Choose on the validation split and report test
accuracies. Give a table with, for each design: parameters, mean qubits per caption, mean two-qubit gates per caption,
and validation and test accuracy on both choices. Part 4 runs these circuits on devices: discuss what each change
costs there, and whether training slows down or stalls as the circuits grow.

*Things to know.* $R_y$ and CNOT are real matrices, so every text and image state is real: half of the state space is
out of reach. With one $s$ qubit, text and image states are points on a circle and their fidelity depends only on the
angle between them. More qubits and layers add expressive power but also post-selected qubits (Part 4), and deep random circuits
tend to have vanishing gradients. The initial angles are drawn uniformly from $[-\pi, \pi]$
(`VQCModel(..., init_scale=...)`).

In [ ]:
# Q7: your code here

**Answer Q7.** *Write your answer here.*

## Part 4. Running the circuits on emulated hardware (45 marks)

So far we have computed $|t\rangle$ exactly. A quantum computer instead *samples*: we prepare the state, measure, and
repeat for many **shots**. Only the shots whose post-selection succeeds are kept, so many shots are thrown away, and
real devices add noise. The code below works, but it is slow and its results are poor: making it fast and reliable
is up to you.

### 4.1 The post-selection budget
A shot is kept only when every post-selected qubit reads 0, which happens with probability
$p_{\text{accept}} = \lVert t\rVert^2$: the norm of the unnormalised text state, which the loss never looks at. The number of *accepted* shots, not the
number of shots, sets the precision of every estimate below.

**Q8 (6 marks).** (a) Compute $p_{\text{accept}} = \lVert t\rVert^2$ for every test caption (`vqc_model` returns the
unnormalised states) and plot it against the number of qubits (log scale), for the baseline and for your model from
Q7. How many shots does the median caption of each size need for 1000 accepted shots? (b) Change the model or its
training to raise $p_{\text{accept}}$ (e.g. a term in the loss) and measure what it gains and what it costs in
accuracy.

In [ ]:
# Q8: your code here

**Answer Q8.** *Write your answer here.*

### 4.2 Sampling the states
Circuits are expensive, so from here on we evaluate only the **image choice on the SVO-Swap rows** whose captions
use only known symbols (lower `N_EMULATE` while you experiment). Each row has three circuits, run separately: the
caption, the positive image and the negative image. We keep the caption's
post-selected shots and compare the output distributions $p$ (caption) and $q$ (image) with
$(\sum_x \sqrt{p_x q_x})^2$.

*Things to know.* Measuring in the computational basis gives $p_x = |\hat t_x|^2$ and $q_x = |v_x|^2$: the phases of
the amplitudes are lost, and $(\sum_x \sqrt{p_x q_x})^2 = (\sum_x |\hat t_x|\,|v_x|)^2 \ge |\langle v|\hat t\rangle|^2$.
To measure $\langle v|\hat t\rangle$ itself, the two states must interfere in one circuit.

In [ ]:
from comp0267.execution.circuits import postselect
from comp0267.execution.emulation import Emulator, fake_backend

N_EMULATE, SHOTS = len(swap_test), 1024
emulated = swap_test[swap_test["vqc"].map(vqc_model.knows)].head(N_EMULATE).reset_index(drop=True)
circuits = []  # per row: caption, positive image, negative image
for _, row in emulated.iterrows():
    circuits.append(to_circuit(row["vqc"], vqc_model.angle_dict()))
    for col in ("pos_image_id", "neg_image_id"):
        image = embeddings[int(row[col])]
        circuits.append(to_circuit(vqc_image_model.recipe, vqc_image_model.angle_dict(image)))
print(f"{len(emulated)} rows, {len(circuits)} circuits")


def overlap(p, q):
    # (sum_x sqrt(p_x q_x))^2 between two count dictionaries, 0 if either kept no shots
    n_p, n_q = sum(p.values()), sum(q.values())
    if n_p == 0 or n_q == 0:
        return 0.0
    return sum(np.sqrt(p[x] / n_p * q.get(x, 0) / n_q) for x in p) ** 2


def choice_accuracy(counts, circuits):
    # counts and circuits in threes (caption, positive, negative), every row is scored (1 right, 0.5 tie, 0 wrong)
    scores, accepted = [], 0
    for i in range(0, len(circuits), 3):
        text = postselect(counts[i], circuits[i])
        accepted += bool(text)
        pos = overlap(text, postselect(counts[i + 1], circuits[i + 1]))
        neg = overlap(text, postselect(counts[i + 2], circuits[i + 2]))
        scores.append((pos > neg) + 0.5 * (pos == neg))
    return np.mean(scores), accepted

In [ ]:
accuracy, kept = choice_accuracy(Emulator().run(circuits, SHOTS, seed=SEED), circuits)
print(f"ideal sampling: accuracy {accuracy:.4f} over {len(emulated)} rows ({kept} with accepted shots)")

**Q9 (10 marks).** (a) Implement a circuit that estimates the fidelity $|\langle v|\hat t\rangle|^2$ itself from
shots, using `to_circuit`, `Emulator` and `postselect`. (b) Compare it with the provided estimator and with the exact fidelities from
`vqc_model` and `vqc_image_model`: accuracy, error, qubits, depth, shots needed and run time. (c) Use what you found to
improve the accuracy of the emulated image choice.

In [ ]:
# Q9: your code here

**Answer Q9.** *Write your answer here.*

### 4.3 Emulating IBM and IQM devices
A *fake device* replays a real device's calibration: its native gates, its qubit connectivity and a noise model of its
gate errors, decoherence and readout errors. We emulate IBM's 133-qubit Torino and IQM's 20-qubit Garnet with noise.

*Things to know.* `Emulator` (like `submit` on real hardware) first transpiles each circuit for the device: it rewrites the gates into the device's
native gates and inserts SWAPs wherever two interacting qubits are not neighbours on the chip (IBM's heavy-hex
lattice has at most 3 neighbours per qubit, Garnet's square lattice at most 4). Every two-qubit gate and every
measurement adds error, and errors on the post-selected qubits also corrupt the post-selection: a wrongly accepted shot is not a
sample of $|\hat t\rangle$ at all. `Emulator` keeps only as many of the device's qubits as the widest circuit needs
(their couplings, gates and noise), so we never simulate the rest of the 133-qubit device. A noisy $n$-qubit circuit
is simulated once, exactly, as a $4^n$-entry density matrix when that is cheaper than one $2^n$-entry state per shot
($2^n \le$ shots), and otherwise once per shot, so the widest captions dominate the time.

In [ ]:
DEVICES = {name: fake_backend(name) for name in ("torino", "garnet")}
for name, device_ in DEVICES.items():
    emulator = Emulator(device_, n_qubits=max(qc.num_qubits for qc in circuits), noisy=True)
    accuracy, kept = choice_accuracy(emulator.run(circuits, SHOTS, seed=SEED), circuits)
    print(f"{name} (noisy): accuracy {accuracy:.4f} over {len(emulated)} rows ({kept} with accepted shots)")

**Q10 (14 marks).** Make the circuits cheaper and more reliable on both devices. Measure first: qubits, transpiled
depth and two-qubit gates (`qiskit.transpile`), accepted shots and accuracy. Ideas: fewer qubits per caption (e.g.
drop or merge boxes such as determiners, reuse qubits with mid-circuit measurement and reset), an ansatz built from
each device's native gates, a better choice of physical qubits (initial layout) or transpiler settings, and shorter
circuits for the images. For each change, report a before/after table on **both** devices and explain why it helps
or not. Credit is for well-motivated, measured improvements, not for the number of ideas.

In [ ]:
# Q10: your code here

**Answer Q10.** *Write your answer here.*

**Q11 (8 marks).** Shots and noise. (a) Plot the noisy accuracy against the number of shots for both devices, and
find how many shots it takes to stop improving. (b) Implement one error-mitigation method (e.g. readout-error
mitigation from calibration circuits) and measure what it gains on both devices: the error of the estimated
fidelities against the exact ones, and the accuracy.

In [ ]:
# Q11: your code here

**Answer Q11.** *Write your answer here.*

**Q12 (7 marks).** Compare the two devices. Relate the differences in transpiled depth, two-qubit gate count and noisy
accuracy to their native gates, connectivity and error rates (`device.target` holds the calibrated gate errors).
For a caption of a given size, which device would you run it on, and why?

In [ ]:
# Q12: your code here

**Answer Q12.** *Write your answer here.*

## Part 5. Discussion (10 marks)

**Q13 (10 marks).** In about 400 words, compare the tensor network with the quantum model using your results: accuracy
on both choices, parameters, cost of training (exact contraction), and cost of inference on a device (qubits, gates,
shots and $1/p_{\text{accept}}$). Using what you measured in Q2, say under which conditions, if any, evaluating such a model on a quantum
computer could be preferable to contracting it classically. Distinguish what your experiments show from what they do
not.

**Answer Q13.** *Write your answer here.*

## 6. Optional: real hardware (not marked)

The circuits run unchanged on real devices. Set `RUN_HARDWARE = True`, choose IBM and/or IQM and paste your tokens below:
`IBM_TOKEN` (and `IBM_INSTANCE`) from IBM Quantum (left empty, the account saved with `QiskitRuntimeService.save_account` is
used), `IQM_TOKEN` from IQM Resonance. Clear the tokens before you share or submit the notebook.
Queues can be long, so start with a few rows. `submit` returns once the jobs are queued and saves their IDs to a file: on Colab
in `COMP0267_jobs` on your Google Drive (Colab erases its own files when the runtime is recycled), elsewhere in `jobs/`.
`retrieve` waits for the results, also after a restart: reconnect to the device and pass it that file.

In [ ]:
RUN_HARDWARE = False
IBM, IQM = True, True

IBM_TOKEN = ''     # IBM Quantum API key
IBM_INSTANCE = ''  # IBM Quantum instance (CRN), may stay empty
IBM_DEVICE = 'ibm_kingston'
IQM_TOKEN = ''     # IQM Resonance API token
IQM_DEVICE = 'garnet'
IQM_URL = 'https://resonance.iqm.tech/'

HARDWARE_ROWS = 5
hardware_circuits = circuits[:3 * HARDWARE_ROWS]  # caption, positive image, negative image per row

In [ ]:
import time

from comp0267.execution.hardware import ibm_backend, iqm_backend, retrieve, submit

if RUN_HARDWARE:
    try:  # on Colab, keep the job files on Google Drive so they outlive the runtime
        from google.colab import drive
        drive.mount('/content/drive')
        JOBS_DIR = '/content/drive/MyDrive/COMP0267_jobs'
    except ImportError:
        JOBS_DIR = 'jobs'

    backends = {}
    if IBM:
        backends['IBM'] = ibm_backend(IBM_DEVICE, token=IBM_TOKEN, instance=IBM_INSTANCE)
    if IQM:
        backends['IQM'] = iqm_backend(IQM_DEVICE, token=IQM_TOKEN, url=IQM_URL)

    jobs = {}  # provider -> job file
    for name, backend in backends.items():
        path = f"{JOBS_DIR}/{backend.name}_{time.strftime('%Y%m%d-%H%M%S')}.json"
        jobs[name] = submit(hardware_circuits, backend, SHOTS, path=path, seed=SEED)

In [ ]:
# waits for the queued jobs. After a restart, reconnect and call retrieve(f'{JOBS_DIR}/<file>.json', backend)
if RUN_HARDWARE:
    for name, path in jobs.items():
        counts = retrieve(path, backends[name])
        accuracy, kept = choice_accuracy(counts, hardware_circuits)
        print(f"{name} {backends[name].name}: accuracy {accuracy:.4f} over {HARDWARE_ROWS} rows ({kept} with accepted shots)")